In [ ]:
from xgboost import XGBClassifier
#from xgboost import XGBRegressor
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import(
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
data = load_breast_cancer()

x = data.data
y = data.target


feature_names = data.feature_names
class_names = data.target_names
print("Dataset shape:",x.shape)
print(feature_names)
print(feature_names.shape)
print("Classes",class_names)

In [ ]:
print(pd.Series(y).value_counts())
pd.Series(y).value_counts(normalize=True) * 100

In [ ]:
#first 70% train,30% tempory
x_train,x_temp,y_train,y_temp = train_test_split(x,y,test_size=0.3,stratify=y,random_state=42)

x_val,x_test,y_val,y_test = train_test_split(x_temp,y_temp,test_size=0.5,stratify=y_temp,random_state=42)

print("Dataset split")
print("Training :",x_train.shape)
print("Validation :",x_val.shape)
print("Test :",x_test.shape) 

In [ ]:
model = XGBClassifier(
    max_depth = 4,
    learning_rate = 0.05,
    n_estimators = 1000,
    eval_metric = 'logloss',
    random_state = 42,
    subsample = 0.8,
    colsample_bytree = 0.8,
    early_stopping_rounds = 30,
    n_jobs = -1
)

In [ ]:
model.fit(
    x_train,y_train,
    eval_set = [(x_train,y_train),(x_val,y_val)]
)

In [ ]:
import sklearn
import xgboost
print("sklearn version",sklearn.__version__)
print("xgboost version",xgboost.__version__)

In [ ]:
result = model.evals_result()
print(result)

In [ ]:
print(f"best iteration: {model.best_iteration}")

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(result['validation_0']['logloss'],label='Training logloss')
plt.plot(result['validation_1']['logloss'],label='Validation logloss')

plt.axvline(
    x=model.best_iteration,
    color = 'red',
    linestyle='--',
    linewidth=2,
    label=f'Early stopping at I teration {model.best_iteration}'
)
plt.xlabel('Bossting round')
plt.ylabel('Log Loss')
plt.title('XGBoost Learning Curves\n(Training vs Validation)')
plt.legend()
plt.grid(True,alpha=0.3)
plt.show()

In [ ]:
y_test_predict = model.predict(x_test)
print(classification_report(y_test,y_test_predict))

In [ ]:
#we save this model using below

model.save_model("breast_canser_xgboost.json")
print("model train suceesfuly")

In [ ]:
loaded = XGBClassifier()
loaded.load_model("breast_canser_xgboost.json")

In [ ]:
print(x[1])
print(y[1])

predicted_value = loaded.predict(x[[1]])
print(predicted_value[0])